# BIOMASS L1 Orthorectification — GCP-Based Geolocation with Delaunay Interpolation

**Objective**: Orthorectify ESA BIOMASS L1 SCS (slant-range complex) SAR data using Ground Control Point (GCP) geolocation.

## Overview

**BIOMASS L1 geolocation** differs from SICD:
- No analytical R/Rdot model
- Instead: **Sparse GCP grid** (lat/lon/height at specific row/col)
- Geolocation uses **Delaunay triangulation** to interpolate between GCPs

This notebook demonstrates:
1. Loading BIOMASS L1 SCS products
2. GCP-based geolocation (`GCPGeolocation`)
3. Pauli decomposition in slant range
4. Orthorectification to UTM grid

## Theory

### GCP Grid Interpolation

BIOMASS provides GCPs at regular intervals (e.g., every 100 pixels):
$$\text{GCP}_{i,j} = (row_i, col_j, lat, lon, h)$$

**Delaunay triangulation** constructs triangles connecting GCPs. For any pixel $(r, c)$:
1. Find the triangle containing $(r, c)$
2. Compute barycentric coordinates $(\alpha, \beta, \gamma)$
3. Interpolate: $lat = \alpha \cdot lat_1 + \beta \cdot lat_2 + \gamma \cdot lat_3$

### Quad-Pol Pauli Decomposition

BIOMASS is **quad-pol** (HH, HV, VH, VV). Pauli RGB:
- **R** (red): $|HH - VV|$ (double-bounce)
- **G** (green): $|HV + VH|$ (volume scattering)
- **B** (blue): $|HH + VV|$ (surface scattering)

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | BIOMASS reader via factory (`get_reader('biomass', ...)`) |
| `grdl.geolocation` | `create_geolocation()` auto-detects GCP geolocation |
| `grdl.image_processing` | `PauliDecomposition` |
| `grdl.image_processing.ortho` | `orthorectify()`, `UTMGrid` |
| `napari` | Interactive visualization |

## Preamble — Version Check

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path
import gc
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.geolocation.chip import ChipGeolocation
from grdl.image_processing.decomposition import PauliDecomposition
from grdl.image_processing.ortho import orthorectify, UTMGrid
from grdl.contrast import PercentileStretch

## Configuration — User Paths

In [ ]:
# Path to BIOMASS L1 SCS product directory
BIOMASS_PATH = Path("/data/sar/biomass/BIO_S3_SCS__1S_20260131T221949_20260131T222010_T_G01_M01_C10_T043_F285_01_DM55O5")

# Validate path
assert BIOMASS_PATH.exists(), f"BIOMASS product not found: {BIOMASS_PATH}"

print(f"✓ BIOMASS product: {BIOMASS_PATH.name}")

## Step 1: Load BIOMASS Metadata and Quad-Pol Chip

In [ ]:
with get_reader('biomass', BIOMASS_PATH) as reader:
    meta = reader.metadata
    shape = reader.get_shape()
    rows, cols = shape[0], shape[1]
    
    print(f"Image size: {rows} × {cols}")
    print(f"Polarizations: {meta.polarizations}")
    print(f"GCPs: {len(meta.gcps) if meta.gcps else 0} control points")
    
    # Extract center chip (4096×4096 or smaller)
    chip_size = min(4096, rows, cols)
    r_start = (rows - chip_size) // 2
    c_start = (cols - chip_size) // 2
    
    # Read quad-pol cube (bands: 0=HH, 1=HV, 2=VH, 3=VV)
    quad_pol = reader.read_chip(
        row_start=r_start, row_end=r_start+chip_size,
        col_start=c_start, col_end=c_start+chip_size,
        bands=[0, 1, 2, 3]  # All pols
    )  # (4, rows, cols)

print(f"\nExtracted chip: {quad_pol.shape}")

## Step 2: Pauli Decomposition in Slant Range

In [ ]:
# Apply Pauli decomposition to quad-pol cube
# Unpack bands: 0=HH, 1=HV, 2=VH, 3=VV
pauli = PauliDecomposition()
components = pauli.decompose(
    quad_pol[0],  # HH
    quad_pol[1],  # HV
    quad_pol[2],  # VH
    quad_pol[3],  # VV
)
pauli_rgb, _ = pauli.to_rgb(components)  # Returns (rgb, metadata)
pauli_rgb = np.moveaxis(pauli_rgb, 0, -1)  # (3, rows, cols) → (rows, cols, 3)

print(f"Pauli RGB: {pauli_rgb.shape}")

## Step 3: Create GCP Geolocation

In [ ]:
with get_reader('biomass', BIOMASS_PATH) as reader:
    # Auto-detect GCP geolocation
    geo_full = create_geolocation(reader)
    
    # Wrap with chip offset
    geo = ChipGeolocation(
        geolocation=geo_full,
        row_offset=r_start,
        col_offset=c_start,
        shape=(chip_size, chip_size),
    )

print(f"✓ Geolocation: {type(geo_full).__name__}")
print(f"  GCP count: {len(geo_full.gcps) if hasattr(geo_full, 'gcps') else 'N/A'}")

## Step 4: Define UTM Output Grid

In [ ]:
# UTM grid from geolocation footprint
output_grid = UTMGrid.from_geolocation(
    geolocation=geo,
    pixel_size_m=5.0,  # BIOMASS ~5m pixel spacing
    margin_m=0.0,
)

print(f"UTM Grid: Zone {output_grid.zone} {'N' if output_grid.north else 'S'}")
print(f"  Size: {output_grid.rows} × {output_grid.cols}")

## Step 5: Orthorectify Pauli RGB

In [ ]:
print("Orthorectifying Pauli RGB (3 bands)...")

# Orthorectify each RGB channel separately
ortho_r = orthorectify(
    geolocation=geo,
    source_array=pauli_rgb[:,:,0],
    output_grid=output_grid,
    interpolation='bilinear'
).data

ortho_g = orthorectify(
    geolocation=geo,
    source_array=pauli_rgb[:,:,1],
    output_grid=output_grid,
    interpolation='bilinear'
).data

ortho_b = orthorectify(
    geolocation=geo,
    source_array=pauli_rgb[:,:,2],
    output_grid=output_grid,
    interpolation='bilinear'
).data

ortho_pauli_rgb = np.stack([ortho_r, ortho_g, ortho_b], axis=-1)

print(f"✓ Orthorectified Pauli RGB: {ortho_pauli_rgb.shape}")

## Step 6: Visualization — Slant vs Ground Pauli

In [ ]:
# Apply percentile stretch to each RGB channel for display
# PercentileStretch ignores NaN when computing percentiles (uses np.nanpercentile)
stretch = PercentileStretch(plow=2.0, phigh=98.0)

# Stretch slant range Pauli RGB (per-channel)
slant_r = stretch.apply(pauli_rgb[:,:,0])
slant_g = stretch.apply(pauli_rgb[:,:,1])
slant_b = stretch.apply(pauli_rgb[:,:,2])
slant_display = np.stack([slant_r, slant_g, slant_b], axis=-1)

# Stretch ortho RGB (with NaN still present - stretch handles it correctly)
# Create separate stretch instances for ortho channels
ortho_r_stretch = PercentileStretch(plow=2.0, phigh=98.0)
ortho_g_stretch = PercentileStretch(plow=2.0, phigh=98.0)
ortho_b_stretch = PercentileStretch(plow=2.0, phigh=98.0)

ortho_r_display = ortho_r_stretch.apply(ortho_pauli_rgb[:,:,0])
ortho_g_display = ortho_g_stretch.apply(ortho_pauli_rgb[:,:,1])
ortho_b_display = ortho_b_stretch.apply(ortho_pauli_rgb[:,:,2])

# Diagnostic: check data ranges
print(f"\nOrtho R channel: [{np.nanmin(ortho_r_display):.3f}, {np.nanmax(ortho_r_display):.3f}]")
print(f"Ortho G channel: [{np.nanmin(ortho_g_display):.3f}, {np.nanmax(ortho_g_display):.3f}]")
print(f"Ortho B channel: [{np.nanmin(ortho_b_display):.3f}, {np.nanmax(ortho_b_display):.3f}]")
print(f"Valid pixels: {np.sum(~np.isnan(ortho_r_display))} / {ortho_r_display.size}")

# Fill NaN with 0 only AFTER stretching (for napari display)
ortho_r_display = np.nan_to_num(ortho_r_display, nan=0.0)
ortho_g_display = np.nan_to_num(ortho_g_display, nan=0.0)
ortho_b_display = np.nan_to_num(ortho_b_display, nan=0.0)

ortho_display = np.stack([ortho_r_display, ortho_g_display, ortho_b_display], axis=-1)

viewer = napari.Viewer(title="BIOMASS Orthorectification — Pauli RGB")

# Layer 1: Slant-range Pauli RGB (rectangular chip, full coverage)
slant_layer = viewer.add_image(
    slant_display,
    name="Pauli Slant Range (rectangular)",
    rgb=True,
    visible=False,
)

# Layer 2: Orthorectified Pauli RGB (diamond-shaped valid data)
ortho_layer = viewer.add_image(
    ortho_display,
    name="Pauli UTM Ortho (diamond)",
    rgb=True,
    visible=True,
)

# Note: No custom toggle logic - use napari's native layer visibility controls
# Click eye icons in layer list to show/hide layers independently

# Simplify viewer UI - hide unnecessary controls
viewer.window._qt_viewer.dockLayerControls.setVisible(False)  # Hide layer controls dock
viewer.window._qt_viewer.dockConsole.setVisible(False)  # Hide console
try:
    viewer.window._qt_viewer.activityDock.setVisible(False)  # Hide activity dock if present
except AttributeError:
    pass

print("✓ Napari viewer opened")
print("  Layer 1: Pauli RGB — rectangular chip (slant-range geometry)")
print("  Layer 2: Pauli UTM Ortho — diamond shape (georeferenced grid)")
print("\n💡 Use the eye icons in the layer list to show/hide layers.")
print("   You can view both layers simultaneously or toggle between them.")
print("  Black regions in ortho are NaN (outside chip boundary in UTM grid).")
print("\nExecution paused — close the napari window to continue and free memory...")

napari.run()

print("\n✓ Viewer closed — cleaning up memory...")
del quad_pol, pauli, components, pauli_rgb, ortho_r, ortho_g, ortho_b, ortho_pauli_rgb
del stretch, slant_r, slant_g, slant_b, slant_display
del ortho_r_stretch, ortho_g_stretch, ortho_b_stretch
del ortho_r_display, ortho_g_display, ortho_b_display, ortho_display
del geo, geo_full, slant_layer, ortho_layer, viewer
gc.collect()
print("✓ Memory cleanup complete")

## Physical Interpretation

### GCP vs R/Rdot Geolocation

| Method | Accuracy | Speed | Use Case |
|--------|----------|-------|----------|
| **R/Rdot** (SICD) | Exact (cm-level) | Slow (iterative) | High-precision SAR |
| **GCP** (BIOMASS) | Interpolated (m-level) | Fast (triangulation) | GCP-based products |

**GCP limitations**:
- Accuracy degrades between GCP grid points
- Assumes linear interpolation (terrain curvature not modeled)
- No DEM integration in Delaunay interpolation

### Pauli Color Interpretation

In orthorectified Pauli RGB:
- **Blue** (surface): Calm water, roads, bare soil
- **Red** (double-bounce): Buildings, tree trunks + ground
- **Green** (volume): Forest canopy, vegetation
- **Yellow** (R+G): Urban areas (mixed double-bounce + volume)
- **Magenta** (R+B): Oriented urban structures
- **Cyan** (G+B): Wetlands, flooded vegetation

## Summary

Successfully demonstrated:
- ✅ BIOMASS L1 quad-pol loading via IO factory
- ✅ GCP-based geolocation (`GCPGeolocation` via `create_geolocation()`)
- ✅ Pauli decomposition in slant range (before ortho)
- ✅ Multi-band orthorectification (R, G, B channels separately)
- ✅ Side-by-side slant vs ground Pauli visualization

### Key GRDL Patterns
- `get_reader('biomass', path)` for BIOMASS products
- `create_geolocation(reader)` auto-detects GCP geolocation
- Process multi-pol data in slant range before ortho
- Orthorectify each channel separately for RGB output

### Next Steps
- Compare GCP vs R/Rdot accuracy (overlay on reference imagery)
- Orthorectify full scene (not just chip)
- Export to GeoTIFF: `GeoTIFFWriter.write(ortho_pauli_rgb, ...)`
- Validate GCP grid spacing vs ortho errors